# Peak assignments: read a server-side run

This notebook reads and analyzes **peak-centric assignment** results persisted
on the server. A peak assignment run assigns a composition to *every* observed
peak of a sample - database-known targets first (Stage A), then untargeted
composition search (Stage B) - arbitrates a single owner per peak, and files
each assignment into a confidence **tier**:

| Tier | Meaning |
| --- | --- |
| `identified` | Confident assignment |
| `candidate` | Plausible but unconfirmed |
| `below_assignability` | Peak too weak/ambiguous to assign |
| `unassigned` | No acceptable composition found |

Where [09_composition_assignment](09_composition_assignment.ipynb) rolls its
own untargeted assignment client-side (no server run needed), this notebook
reads the server engine's persisted, arbitrated, calibrated results.

> **Prerequisite:** the sample must already have a **completed** assignment
> run, launched from the Mascope app (Sample view). The SDK read surface is
> read-only - it does not trigger runs.

In [ ]:
from mascope_sdk import MascopeClient


mascope = MascopeClient(workspace="My Workspace")

### Pick a sample

List the samples of a batch and pick one. Any sample works as long as it has a
completed assignment run.

In [ ]:
# Adjust these to match your data
batch = "My Batch"

samples = mascope.samples.list(batch=batch)
sample_id = samples.iloc[0]["sample_item_id"]
samples[["sample_item_id", "sample_item_name", "datetime_utc"]].head()

### Run history

A sample can have several runs (e.g. re-assigned after a target-library
update). `list_runs` returns them newest first with status, engine version,
and the configuration each run used.

In [ ]:
runs = mascope.peak_assignments.list_runs(sample_id)
runs

### Read the ledger

`get()` returns the whole run as a single DataFrame - **one row per observed
peak** - paging through the API internally. The run metadata rides along on
`df.attrs["run"]`. Without `run_id` it reads the latest completed run.

In [ ]:
assignments = mascope.peak_assignments.get(sample_id)

run = assignments.attrs["run"]
print(f"Run {run['peak_assignment_run_id']} (engine {run['engine_version']})")
print(f"Completed: {run['peak_assignment_run_utc_completed']}")
print(f"Config: {run['config']}")
print(f"{len(assignments)} peaks")
assignments.head()

### Tier distribution

How much of the sample is explained, and how confidently? Compare peak
*counts* per tier with the share of total *intensity* per tier - a few
identified peaks often carry most of the signal.

In [ ]:
import plotly.express as px
import plotly.io as pio


pio.templates.default = "plotly_dark"  # or "plotly_white"

TIER_ORDER = ["identified", "candidate", "below_assignability", "unassigned"]

tier_stats = (
    assignments.groupby("tier")
    .agg(peaks=("sample_peak_id", "size"), intensity=("sample_peak_intensity", "sum"))
    .reindex(TIER_ORDER)
    .fillna(0)
)
tier_stats["intensity_share"] = tier_stats["intensity"] / tier_stats["intensity"].sum()

fig = px.bar(
    tier_stats.reset_index(),
    x="tier",
    y="peaks",
    hover_data={"intensity_share": ":.1%"},
    title="Peaks per confidence tier",
)
fig.show()

tier_stats

### Source split: database vs untargeted

Each assigned peak records where its winning composition came from:

- `database` - Stage A, the curated target library (the targeted result,
  peak-anchored)
- `untargeted` - Stage B, the untargeted composition search

The split shows how much of the sample the target list already covers and how
much the untargeted stage adds on top.

In [ ]:
assigned = assignments[assignments["assigned_formula"].notna()]

source_split = assigned.groupby(["tier", "source"]).size().rename("peaks").reset_index()

fig = px.bar(
    source_split,
    x="tier",
    y="peaks",
    color="source",
    barmode="group",
    category_orders={"tier": TIER_ORDER},
    title="Assigned peaks by tier and source",
)
fig.show()

### Tier-colored mass-defect map

The classic overview plot: nominal mass vs mass defect, here colored by
confidence tier. Homologous series form visible lines; unassigned or
low-confidence regions stand out immediately.

In [ ]:
import numpy as np


plot_df = assignments.copy()
plot_df["mass_defect"] = plot_df["sample_peak_mz"] - plot_df["sample_peak_mz"].round()
plot_df["log_intensity"] = np.log10(plot_df["sample_peak_intensity"].clip(lower=1))

fig = px.scatter(
    plot_df,
    x="sample_peak_mz",
    y="mass_defect",
    color="tier",
    size="log_intensity",
    category_orders={"tier": TIER_ORDER},
    hover_data=["assigned_formula", "ion_formula", "source", "fit_score"],
    title="Mass-defect map colored by tier",
    labels={"sample_peak_mz": "m/z", "mass_defect": "Mass defect"},
)
fig.show()

### Van Krevelen

Plot H/C vs O/C elemental ratios of the assigned (neutral) formulas. Regions
of the Van Krevelen space correspond to compound classes (e.g. lipids,
carbohydrates, condensed aromatics), so it is a quick chemical fingerprint of
the sample - here split by assignment source.

In [ ]:
import re


def element_counts(formula: str) -> dict[str, int]:
    """Count elements in a simple molecular formula (e.g. 'C6H12O6')."""
    counts: dict[str, int] = {}
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts


# M0 rows only: isotope children repeat the owner's formula.
vk = assigned[assigned["role"] == "M0"].copy()
counts = vk["assigned_formula"].map(element_counts)
vk["C"] = counts.map(lambda c: c.get("C", 0))
vk["H"] = counts.map(lambda c: c.get("H", 0))
vk["O"] = counts.map(lambda c: c.get("O", 0))
vk = vk[vk["C"] > 0]
vk["O/C"] = vk["O"] / vk["C"]
vk["H/C"] = vk["H"] / vk["C"]

fig = px.scatter(
    vk,
    x="O/C",
    y="H/C",
    color="source",
    size=np.log10(vk["sample_peak_intensity"].clip(lower=1)),
    hover_data=["assigned_formula", "tier", "fit_score", "sample_peak_mz"],
    title="Van Krevelen of assigned compositions",
)
fig.show()

### Drill into a contested peak

The ledger rows are a slim projection. For a single assignment, `detail()`
fetches the full record, including the ranked **alternative compositions** the
engine considered and the scoring **provenance** - useful for judging how
contested a `candidate` assignment is.

In [ ]:
candidates = assignments[assignments["tier"] == "candidate"]
if candidates.empty:
    print("No candidate-tier peaks in this run")
else:
    top = candidates.sort_values("sample_peak_intensity", ascending=False).iloc[0]
    full = mascope.peak_assignments.detail(sample_id, top["peak_assignment_id"])

    print(
        f"Peak m/z {top['sample_peak_mz']:.4f} -> {full['assigned_formula']} "
        f"(fit {full['fit_score']:.3f}, P(correct) {full['p_correct']})"
    )
    print("Alternatives considered:")
    for alternative in full["alternatives"] or []:
        print(f"  {alternative}")

### Server-side filters and cross-sample loading

`tier`, `role`, and `source` filter on the server, so narrow reads stay cheap:

```python
identified = mascope.peak_assignments.get(sample_id, tier="identified")
stage_b = mascope.peak_assignments.get(sample_id, source="untargeted")
```

To analyze assignments **across a whole batch** (or several), use the
high-level loader - it reads each sample's latest completed run concurrently
and concatenates them with batch/sample metadata, exactly like `load_peaks`:

```python
assignments = mascope.load_assignments(dataset="My Dataset", batches="My Batch")
assignments.groupby(["sample_item_name", "tier"]).size()
```

Samples without a completed run are skipped (and logged), not assigned on the
fly - launching runs stays in the Mascope app for now.